# Chronological comparison across three games

Use the same eight behavioural inputs in all games. Reserve the most recent fifth of players by first observation. Use earlier players whose labels are complete before the first test prediction to fit the model, calibrate it, and pick the threshold. Low and high activity cutoffs come from training counts. Notebook 08 uses the stricter separate training and validation periods for the Game 1 model-family comparison. Notebook 09 also tests a Game 1 one-event specialist.

The feature stage applies the same registered-account filter as the label stage. The notebook checks that feature counts match label-stage observation counts.

In [1]:
import os
import sys
from pathlib import Path

candidates = (
    [Path(os.environ["CHURN_PROJECT_ROOT"])] if "CHURN_PROJECT_ROOT" in os.environ else
    [Path.cwd() / ".pipeline", Path.cwd(), Path.cwd().parent / ".pipeline", Path.cwd().parent]
)
ROOT = next((candidate.resolve() for candidate in candidates
             if (candidate / "src" / "config.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run 00_setup.ipynb first or point CHURN_PROJECT_ROOT at the pipeline checkout.")
sys.path.insert(0, str(ROOT))



In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, balanced_accuracy_score
from sklearn.metrics import brier_score_loss, roc_auc_score, roc_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from src import config, features, labeling

INPUTS = ["recency_hours", "play_count", "active_days", "active_duration_hours",
          "session_count", "weekend_ratio", "night_ratio", "consecutive_play_ratio"]
BOOTSTRAPS = 2000
SEED = 42

def build_table(game):
    event_file = config.INTERIM_DIR / f"{game}_events.parquet"
    fields = ["player_id", "ts", "authed"]
    if game == "game1":
        fields.append("score")
    events = pd.read_parquet(event_file, columns=fields)
    cohort = labeling.build_cohort(events)
    eligible_event = events.authed.fillna(True)
    excluded = events.loc[~eligible_event, ["player_id", "ts"]]
    legacy_overlap = excluded.merge(cohort[["player_id", "op_end"]], on="player_id")
    mismatches = int(legacy_overlap.loc[legacy_overlap.ts < legacy_overlap.op_end,
                                        "player_id"].nunique())
    events = events.loc[eligible_event]
    for field in ("score", "won", "action", "fail_state", "duration", "price", "platform", "device"):
        if field not in events:
            events[field] = np.nan
    player = features.build_features(events, cohort, config.OBSERVATION_PERIOD_DAYS)
    data = player.merge(cohort[["player_id", "cp_end", "n_events_op"]],
                        on="player_id", validate="1:1")
    assert np.array_equal(data.play_count, data.n_events_op)
    return data.sort_values("first_ts", kind="stable"), mismatches

def temporal_split(table):
    idx = table.first_ts
    test_start = idx.iloc[int(.8 * len(table))]
    first_test_prediction = test_start + config.OBSERVATION_PERIOD_DAYS * labeling.DAY
    earlier = table[(idx < test_start) & (table.cp_end <= first_test_prediction)]
    val_start = earlier.first_ts.iloc[int(.75 * len(earlier))]
    train = earlier[earlier.first_ts < val_start]
    val = earlier[earlier.first_ts >= val_start]
    test = table[idx >= test_start]
    assert earlier.cp_end.max() <= first_test_prediction
    assert train.first_ts.max() < val.first_ts.min() < test.first_ts.min()
    assert len(set(train.player_id) | set(val.player_id) | set(test.player_id)) == sum(
        len(part) for part in (train, val, test))
    return train, val, test

def fit(train, val, test, columns):
    model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                          LogisticRegression(max_iter=2000, class_weight="balanced"))
    model.fit(train[columns], train.churn)
    val_raw = model.predict_proba(val[columns])[:, 1]
    test_raw = model.predict_proba(test[columns])[:, 1]
    calibrator = LogisticRegression(C=1000, max_iter=1000).fit(val_raw[:, None], val.churn)
    validation_score = calibrator.predict_proba(val_raw[:, None])[:, 1]
    test_score = calibrator.predict_proba(test_raw[:, None])[:, 1]
    fpr, tpr, thresholds = roc_curve(val.churn, validation_score, drop_intermediate=False)
    threshold = thresholds[1:][np.argmax((tpr-fpr)[1:])]
    return test_score, float(threshold)

def score(y, p, threshold):
    retained = y == 0
    return {"n": len(y), "retained": int(retained.sum()),
            "retained_correct": int((retained & (p < threshold)).sum()),
            "roc_auc": float(roc_auc_score(y, p)),
            "balanced_accuracy": float(balanced_accuracy_score(y, p >= threshold)),
            "brier": float(brier_score_loss(y, p)),
            "average_precision_retained": float(average_precision_score(1-y, 1-p)),
            "threshold": threshold}

def auc_interval(y1, p1, y2, p2, paired=False):
    rng = np.random.default_rng(SEED if paired else SEED + 1)
    cells1 = [np.flatnonzero(y1 == value) for value in (0, 1)]
    cells2 = [np.flatnonzero(y2 == value) for value in (0, 1)]
    assert all(len(cell) for cell in cells1 + cells2)

    def sample(cells):
        return np.concatenate([rng.choice(cell, len(cell), replace=True) for cell in cells])

    def auc_ordered(scores, group):
        neg, pos = map(len, group)
        return (rankdata(scores)[neg:].sum() - pos * (pos + 1) / 2) / (pos * neg)

    changes = []
    for _ in range(BOOTSTRAPS):
        i = sample(cells1)
        j = i if paired else sample(cells2)
        changes.append(auc_ordered(p1[i], cells1) - auc_ordered(p2[j], cells2))
    return np.quantile(changes, [.025, .975]).tolist()

def describe_partition(p):
    return {"players": len(p), "retained": int((p.churn == 0).sum()),
            "first": str(pd.to_datetime(p.first_ts.min(), unit="s").date()),
            "last": str(pd.to_datetime(p.first_ts.max(), unit="s").date())}

def experiment(game):
    table, legacy_count_mismatches = build_table(game)
    train, val, test = temporal_split(table)
    y = test.churn.to_numpy()
    common, threshold = fit(train, val, test, INPUTS)
    recency, recency_threshold = fit(train, val, test, ["recency_hours"])
    light_max = int(train.play_count.quantile(.25, interpolation="higher"))
    heavy_min = max(light_max + 1, int(train.play_count.quantile(.75, interpolation="higher")))
    light = test.play_count.le(light_max).to_numpy()
    heavy = test.play_count.ge(heavy_min).to_numpy()
    result = {
        "legacy_count_mismatches": legacy_count_mismatches,
        "cohort": len(table), "omitted": len(table) - len(train) - len(val) - len(test),
        "train": describe_partition(train), "validation": describe_partition(val),
        "test": describe_partition(test),
        "common": score(y, common, threshold), "recency": score(y, recency, recency_threshold),
        "auc_difference_interval": auc_interval(y, common, y, recency, paired=True),
        "constant_brier": float(brier_score_loss(y, np.full(len(y), train.churn.mean()))),
        "light_max": light_max, "heavy_min": heavy_min,
        "light": score(y[light], common[light], threshold),
        "heavy": score(y[heavy], common[heavy], threshold),
        "engagement_gap_interval": auc_interval(y[heavy], common[heavy], y[light], common[light]),
    }
    if game == "game1":
        # A separate score-and-timing model for players with only one observed event.
        light_train = train[train.play_count.eq(1)]
        light_val = val[val.play_count.eq(1)]
        one = test.play_count.eq(1).to_numpy()
        targeted, targeted_threshold = fit(
            light_train, light_val, test.loc[one], ["score_mean", "weekend_ratio", "night_ratio"])
        result["one_event_common"] = score(y[one], common[one], threshold)
        result["one_event_specialist"] = score(y[one], targeted, targeted_threshold)
        result["one_event_specialist_auc_interval"] = auc_interval(
            y[one], targeted, y[one], common[one], paired=True)
    return result

In [3]:
import json
results = {}
for game in config.GAMES:
    print(f"Evaluating {game}", flush=True)
    results[game] = experiment(game)
    print(json.dumps(results[game], indent=2), flush=True)
    (ROOT / "three_game_temporal_results.json").write_text(json.dumps(results, indent=2) + "\n", encoding="utf-8")


Evaluating game1


  cohort: 25,956 players -> 25,422 after left guard -> 25,121 after right guard -> 25,121 with OP activity | churn rate 92.6%


{
  "legacy_count_mismatches": 0,
  "cohort": 25121,
  "omitted": 589,
  "train": {
    "players": 14630,
    "retained": 1232,
    "first": "2015-01-18",
    "last": "2015-09-03"
  },
  "validation": {
    "players": 4877,
    "retained": 284,
    "first": "2015-09-03",
    "last": "2015-12-27"
  },
  "test": {
    "players": 5025,
    "retained": 307,
    "first": "2016-01-06",
    "last": "2016-05-09"
  },
  "common": {
    "n": 5025,
    "retained": 307,
    "retained_correct": 172,
    "roc_auc": 0.7732296989973945,
    "balanced_accuracy": 0.7143185085050945,
    "brier": 0.049983026676469866,
    "average_precision_retained": 0.316229034004478,
    "threshold": 0.9386953417538958
  },
  "recency": {
    "n": 5025,
    "retained": 307,
    "retained_correct": 206,
    "roc_auc": 0.7811072156948301,
    "balanced_accuracy": 0.7363103120214634,
    "brier": 0.05051497790057389,
    "average_precision_retained": 0.30152953952840245,
    "threshold": 0.9688299119780984
  },
  "auc_di

Evaluating game2


  cohort: 79,436 players -> 79,435 after left guard -> 79,410 after right guard -> 79,410 with OP activity | churn rate 75.5%


{
  "legacy_count_mismatches": 0,
  "cohort": 79410,
  "omitted": 46253,
  "train": {
    "players": 12956,
    "retained": 3665,
    "first": "2014-07-02",
    "last": "2014-07-10"
  },
  "validation": {
    "players": 4319,
    "retained": 1271,
    "first": "2014-07-10",
    "last": "2014-07-11"
  },
  "test": {
    "players": 15882,
    "retained": 2645,
    "first": "2014-07-21",
    "last": "2014-09-15"
  },
  "common": {
    "n": 15882,
    "retained": 2645,
    "retained_correct": 1091,
    "roc_auc": 0.6817159125913459,
    "balanced_accuracy": 0.6411176896746289,
    "brier": 0.1300835937504124,
    "average_precision_retained": 0.3845547966206422,
    "threshold": 0.6363339585990933
  },
  "recency": {
    "n": 15882,
    "retained": 2645,
    "retained_correct": 1136,
    "roc_auc": 0.6741113619625805,
    "balanced_accuracy": 0.6423719073519791,
    "brier": 0.1332094148455142,
    "average_precision_retained": 0.3574035919662959,
    "threshold": 0.6491144303011152
  },
 

Evaluating game3


  cohort: 26,868 players -> 25,705 after left guard -> 24,128 after right guard -> 24,128 with OP activity | churn rate 60.0%


{
  "legacy_count_mismatches": 2371,
  "cohort": 24128,
  "omitted": 1132,
  "train": {
    "players": 13627,
    "retained": 6084,
    "first": "2015-05-30",
    "last": "2015-08-30"
  },
  "validation": {
    "players": 4543,
    "retained": 1531,
    "first": "2015-08-30",
    "last": "2015-10-15"
  },
  "test": {
    "players": 4826,
    "retained": 1616,
    "first": "2015-10-25",
    "last": "2015-12-10"
  },
  "common": {
    "n": 4826,
    "retained": 1616,
    "retained_correct": 1181,
    "roc_auc": 0.8114252722001172,
    "balanced_accuracy": 0.7663429952808365,
    "brier": 0.1494277946706094,
    "average_precision_retained": 0.7393039007281825,
    "threshold": 0.6741059718751267
  },
  "recency": {
    "n": 4826,
    "retained": 1616,
    "retained_correct": 1209,
    "roc_auc": 0.8180172573332098,
    "balanced_accuracy": 0.7667509099040746,
    "brier": 0.15003504547659052,
    "average_precision_retained": 0.7346572685449326,
    "threshold": 0.7352974553196328
  },
 